# News fetching from News #
> Lets the user input a stock symbol or leave it blank to see top S&P 500 news.
> Displays the top 5 recent stock market news.
> Highlights the mentioned S&P 500 stocks with current and yesterday's prices in each news article.
> Maintains a chat-like interface with a friendly personality

In [ ]:
!pip install yfinance

In [ ]:
# ========================================
# stock_news_bot_with_prices.py
# ========================================
import pandas as pd
import os
import requests
import datetime
import re
from collections import Counter
import gradio as gr
import yfinance as yf

# ------------------------------
# Config
# ------------------------------
API_KEY = "10c68261d5ab453782221107e727559f" 
SP500_CSV_PATH = "data/sp500_tickers.csv"
PAGE_SIZE = 50
NEWS_QUERY = "stock market OR finance OR S&P 500"

# ------------------------------
# Ensure S&P 500 CSV exists
# ------------------------------
def create_sp500_csv_if_missing(csv_path=SP500_CSV_PATH):
    if not os.path.exists(csv_path):
        print("S&P 500 CSV not found. Creating it now...")
        url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36"
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        df = pd.read_html(response.text, header=0)[0]
        # Keep only symbol + company name (column may vary)
        df = df[['Symbol', 'Security']] if 'Security' in df.columns else df[['Symbol', df.columns[1]]]
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        df.to_csv(csv_path, index=False)
        print(f"S&P 500 tickers saved to {csv_path}")

create_sp500_csv_if_missing()

# Load tickers and company names
df_sp500 = pd.read_csv(SP500_CSV_PATH)
# Detect company name column dynamically
company_col = None
for col in df_sp500.columns:
    if "security" in col.lower() or "name" in col.lower():
        company_col = col
        break

if company_col is None:
    print("Warning: Could not find company name column, using ticker only.")
    TICKER_TO_NAME = {t: t for t in df_sp500["Symbol"]}
else:
    TICKER_TO_NAME = dict(zip(df_sp500["Symbol"], df_sp500[company_col]))

SP500_TICKERS = list(TICKER_TO_NAME.keys())

# ------------------------------
# Helper: get stock price
# ------------------------------
def get_stock_prices(ticker):
    try:
        stock_obj = yf.Ticker(ticker)
        hist = stock_obj.history(period="2d")
        if len(hist) >= 2:
            yesterday_close = round(hist['Close'][-2], 2)
            today_price = round(hist['Close'][-1], 2)
        elif len(hist) == 1:
            yesterday_close = "N/A"
            today_price = round(hist['Close'][-1], 2)
        else:
            yesterday_close = today_price = "N/A"
    except Exception:
        yesterday_close = today_price = "N/A"
    return today_price, yesterday_close

# ------------------------------
# FUNCTION 1: Fetch news for a specific stock
# ------------------------------
def fetch_news_for_stock(stock_symbol, page_size=PAGE_SIZE):
    stock_symbol = stock_symbol.upper()
    from_date = (datetime.datetime.now() - datetime.timedelta(days=3)).strftime("%Y-%m-%d")
    url = f"https://newsapi.org/v2/everything?q={NEWS_QUERY}&from={from_date}&sortBy=publishedAt&pageSize={page_size}&apiKey={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        return [f"Error fetching news: {e}"]
    
    data = response.json()
    articles = data.get("articles", [])
    filtered_news = []
    
    for article in articles:
        title = article.get("title", "")
        description = article.get("description", "")
        content = f"{title} {description}"
        if re.search(rf"\b{stock_symbol}\b", content, re.IGNORECASE):
            company_name = TICKER_TO_NAME.get(stock_symbol, "Unknown Company")
            today_price, yesterday_close = get_stock_prices(stock_symbol)
            source = article.get("source", {}).get("name", "Unknown")
            published_at = article.get("publishedAt", "Unknown")
            summary = (
                f"Title: {title}\n"
                f"Description: {description}\n"
                f"Source: {source}\n"
                f"Published: {published_at}\n"
                f"{stock_symbol} ({company_name}) Price: ${today_price}, Yesterday Close: ${yesterday_close}"
            )
            filtered_news.append(summary)
    
    if not filtered_news:
        filtered_news = [f"No recent news found for {stock_symbol}."]
    
    return filtered_news

# ------------------------------
# FUNCTION 2: Fetch top N news & affected stocks with prices
# ------------------------------
def fetch_top_news_with_stocks(top_n=10):
    from_date = (datetime.datetime.now() - datetime.timedelta(days=3)).strftime("%Y-%m-%d")
    url = f"https://newsapi.org/v2/everything?q={NEWS_QUERY}&from={from_date}&sortBy=publishedAt&pageSize={top_n}&apiKey={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        return [f"Error fetching news: {e}"]
    
    data = response.json()
    articles = data.get("articles", [])
    if not articles:
        return ["No news found in the last 3 days."]
    
    news_summaries = []
    for article in articles:
        title = article.get("title", "")
        description = article.get("description", "")
        content = f"{title} {description}"
        source = article.get("source", {}).get("name", "Unknown")
        published_at = article.get("publishedAt", "Unknown")
        
        mentioned_stocks = [ticker for ticker in SP500_TICKERS if ticker.upper() in content.upper()]
        mentioned_stocks = mentioned_stocks[:10]  # limit 10
        
        prices_info = []
        for ticker in mentioned_stocks:
            company_name = TICKER_TO_NAME.get(ticker, "Unknown Company")
            today_price, yesterday_close = get_stock_prices(ticker)
            prices_info.append(f"{ticker} ({company_name}): ${today_price} (Yesterday: ${yesterday_close})")
        prices_str = ", ".join(prices_info) if prices_info else "None"
        
        summary = (
            f"Title: {title}\n"
            f"Description: {description}\n"
            f"Source: {source}\n"
            f"Published: {published_at}\n"
            f"Affected Stocks & Prices: {prices_str}"
        )
        news_summaries.append(summary)
    
    return news_summaries

# ------------------------------
# Restricted topics guardrail
# ------------------------------
RESTRICTED_TOPICS = [
    "cat", "cats", "dog", "dogs", 
    "horoscope", "zodiac", 
    "taylor swift"
]

def contains_restricted_topic(text):
    text_lower = text.lower()
    for topic in RESTRICTED_TOPICS:
        if topic in text_lower:
            return True
    return False

# ------------------------------
# Gradio Callback with Guardrails
# ------------------------------
def stock_news_bot(stock_symbol):
    stock_symbol = stock_symbol.upper().strip()
    
    # Guardrail: Check for restricted topics
    if contains_restricted_topic(stock_symbol):
        return "⚠️ Sorry, this topic is restricted and cannot be discussed."
    
    # Guardrail: Prevent access/modification of system prompt
    if any(word in stock_symbol.lower() for word in ["system prompt", "modify prompt"]):
        return "⚠️ Access denied: You cannot view or modify the system prompt."

    # Proceed normally
    if stock_symbol == "":
        top_news = fetch_top_news_with_stocks(top_n=10)
        return "\n\n".join(top_news)
    else:
        return "\n\n".join(fetch_news_for_stock(stock_symbol))

# ------------------------------
# Gradio Interface
# ------------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 📈 Stock NewsBot with Prices & Company Names")
    gr.Markdown("Type a stock symbol (like `AAPL`) or leave blank to see top news with affected stocks, their company names, and prices.")
    stock_input = gr.Textbox(label="Stock Symbol", placeholder="e.g., TSLA")
    output_area = gr.Textbox(label="News Output", lines=25)
    stock_input.submit(stock_news_bot, inputs=stock_input, outputs=output_area)

demo.launch()